In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Import required Dash components for layout and interactivity
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Import standard Python libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Import the CRUD Python module developed in Project One
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# Define database credentials
username = "aacuser"
password = "Password123!"

# Instantiate the CRUD class (connection handled internally)
db = AnimalShelter()

# Retrieve all records from the MongoDB collection
df = pd.DataFrame.from_records(db.read({}))

# Remove the MongoDB ObjectID column to prevent issues with Dash
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

# Initialize the Dash application
app = JupyterDash(__name__)

# Load and encode the Grazioso Salvare logo
image_filename = 'grazioso.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Define the layout of the dashboard
app.layout = html.Div([

    # Dashboard title with unique identifier
    html.Center(html.B(html.H1('Kenya Craw Dashboard'))),

    # Display the logo
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), width='200px'),

    html.Hr(),

    # Interactive filtering options using radio buttons
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Wilderness Rescue', 'value': 'wilderness'},
                {'label': 'Disaster/Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset'
        )
    ]),

    html.Hr(),

    # Data table displaying records from MongoDB
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),

        # Enable interactivity features for usability
        row_selectable="single",
        selected_rows=[0],
        page_size=10,
        sort_action="native",
        filter_action="native"
    ),

    html.Br(),
    html.Hr(),

    # Container for graph and map displayed side-by-side
    html.Div(className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ])
])


#############################################
# Controller: Filtering Logic
#############################################

# Update the data table based on selected filter option
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):

    # Define queries based on rescue type
    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Labrador|Retriever|Newfoundland"}
        }

    elif filter_type == 'wilderness':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "German Shepherd|Doberman|Rottweiler"}
        }

    else:
        # Reset returns all records
        query = {}

    # Retrieve filtered data
    dff = pd.DataFrame.from_records(db.read(query))

    # Remove ObjectID if present
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


#############################################
# Controller: Graph Update
#############################################

# Update chart based on filtered table data
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    # Prevent errors if no data is available
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Create a histogram showing breed distribution
    return [
        dcc.Graph(
            figure=px.histogram(dff, x="breed", title="Breed Distribution")
        )
    ]


#############################################
# Controller: DataTable Styling
#############################################

# Highlight selected columns in the data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


#############################################
# Controller: Map Update
#############################################

# Update geolocation map based on selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    # Handle cases where data is not yet available
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Default to first row if no selection
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Generate map with marker for selected animal
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[dff.iloc[row,13], dff.iloc[row,14]],
                             children=[
                                 dl.Tooltip(dff.iloc[row,4]),
                                 dl.Popup([
                                     html.H1("Animal Name"),
                                     html.P(dff.iloc[row,9])
                                 ])
                             ])
               ])
    ]


# Run the dashboard application
app.run_server()

Dash app running on https://marvinloyal-roadnormal-3000.codio.io/proxy/8050/
